# FabricDefectHub — 云端 GPU 训练（Notebook 版）

这个 notebook 是 `fdh train` 的**同一条代码路径**，不是平行实现：每个单元格调用的都是
`fabric_defect_hub` 的门面层（`fdh.load_config` / `fdh.train` / `fdh.evaluate`），
和 CLI 走的函数完全一样。所以在这里跑出来的权重、指标、provenance 记录，与
`fdh train` 产出的没有任何区别——notebook 只提供更方便的分步执行和参数改动。

**框架**：本项目全栈 PyTorch，**不含也不需要 TensorFlow**（六个后端
ultralytics / torchvision / anomalib / dinomaly / moeclip / mambaad 的上游全是 PyTorch）。

**进度**：项目自己写的四个训练循环（torchvision / dinomaly / moeclip / mambaad）通过
`core.progress` 每隔几秒打印一行纯文本进度；anomalib 走 Lightning 自带进度条，
Ultralytics 走它自己的表格输出。全部会直接显示在下面的单元格输出里。

顺序：**1 环境 → 2 体检 → 3 参数 → 4 冒烟 → 5 正式训练 → 6 曲线 → 7 评测**。

## 1. 环境

把工作目录切到仓库根目录（`data/` 符号链接、`configs/`、`artifacts/` 都是相对它解析的），
并设置 HuggingFace 镜像——anomalib 的 backbone 从 `huggingface.co` 下载，
国内云主机通常直连不通（见 `docs/cloud_training_runbook.md` §4）。

In [ ]:
import os, sys, pathlib

PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")  # anomalib backbone 下载
os.environ.setdefault("FDH_PROGRESS_INTERVAL", "5")            # 进度行间隔（秒）

import torch
print("项目根目录 :", PROJECT_ROOT)
print("torch      :", torch.__version__)
print("CUDA 可用  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
    print("显存       : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. 这台机器上什么能跑

`fdh doctor` 的可视化版本：每个后端框架是否装上、会用哪个数据集、为什么不能跑。
下面第二段列出 `data/` 里每个数据集软链接的真实状态（**已挂载 / 断链 / 缺失**）——
断链（指向本机不存在的路径）是从本地把仓库同步到云端时最常见的一种坑。

In [ ]:
import importlib
from fabric_defect_hub.cli import _run_doctor

for backend, entry in _run_doctor()["backends"].items():
    mark = "✅" if entry["trainable_now"] else "❌"
    print(f"{mark} {backend:12s} dataset={entry.get('dataset')!s:14s} {entry['reason']}")

print("\n-- data/ 挂载状态 --")
importlib.import_module("fabric_defect_hub.datasets")
from fabric_defect_hub.core.availability import root_is_staged
from fabric_defect_hub.training import DEFAULT_DATASET_ROOTS

for name, root in sorted(DEFAULT_DATASET_ROOTS.items()):
    path = pathlib.Path(root)
    if root_is_staged(root):
        state = "✅ 已挂载"
    elif path.is_symlink():
        state = f"⚠️  断链 -> {os.readlink(path)}"
    else:
        state = "— 未挂载"
    print(f"{name:16s} {root:28s} {state}")

## 3. 参数

只改这一格。`MODEL` 接受 `fdh train` 认识的一切写法：模型关键字（`"stfpm"`、`"yolov8n"`、
`"patchcore"`）、配置文件名、或完整路径。`fdh.list_models()` 打印全部可选项。

`RUN_LENGTH` 的单位随后端而变——anomalib/YOLO/torchvision/MoECLIP 数 epoch，
Dinomaly/MambaAD 数优化器迭代次数。下一格会告诉你这次算的是哪个。

In [ ]:
MODEL        = "stfpm"        # 见 fabric_defect_hub.list_models()
DATASET      = "zju-leaper"   # 见 fabric_defect_hub.list_datasets()
ACCELERATOR  = "gpu"          # "gpu" | "cpu" | "auto"（配置里默认写死 cpu，云端必须显式给 gpu）
NUM_SAMPLES  = 300            # 训练取样张数
RUN_LENGTH   = 100            # epoch 或 iteration，见下
EXTRA_SET    = {}             # 任意点分路径覆盖，如 {"train.model_kwargs.lr": 0.0005}

import fabric_defect_hub as fdh
print(fdh.list_models())

In [ ]:
from fabric_defect_hub.training import RUN_LENGTH_KEYS, device_override

def build_config(run_length=RUN_LENGTH, num_samples=NUM_SAMPLES, **extra_set):
    """把上面的参数翻成一个 RunConfig。每个覆盖层都是 `fdh train` 自己用的那一层。"""
    cfg = fdh.load_config(MODEL, dataset=DATASET, num_samples=num_samples)
    length_key, unit = RUN_LENGTH_KEYS[cfg.backend]
    overrides = {length_key: run_length}
    overrides.update(device_override(cfg.backend, ACCELERATOR))  # 后端各自的设备写法
    overrides.update(EXTRA_SET)
    overrides.update(extra_set)
    return cfg.with_set(**overrides), unit

cfg, unit = build_config()
print("后端        :", cfg.backend)
print("解析到的配置:", cfg.config_path)
print("变体        :", cfg.variant)
print(f"运行长度    : {RUN_LENGTH} {unit}")
print("覆盖层      :", cfg.set_overrides)

## 4. 冒烟：8 张图、1 个单位

先确认整条链路（读数据 → 构模型 → 存 checkpoint → 评测）在这台机器上是通的，再投入真训练。
`publish=False` 很重要：它不会覆盖前端正在用的那份已发布权重。

In [ ]:
smoke_cfg, _ = build_config(run_length=1, num_samples=8)
smoke = fdh.train(smoke_cfg, publish=False)
print("\n冒烟通过 ->", smoke.result.registered_artifact.path)
print("指标      :", smoke.result.metrics)

## 5. 正式训练

进度会直接打在输出区（`[fdh] ...` 是本项目自己的循环，其余是 Lightning / Ultralytics 自带的）。

`publish=True`（默认）会把权重复制到 `artifacts/models/published/<Key>.ckpt`，
也就是 Web UI 读取的固定位置——只有目录内模型（`fdh.list_pretrained()`）才有这一步。

In [ ]:
cfg, unit = build_config()
run = fdh.train(cfg)

print("\n注册权重:", run.result.registered_artifact.path)
print("发布路径:", run.published_path)
print("指标    :", run.result.metrics)

## 6. 训练曲线

YOLO 写 `results.csv`，torchvision / Dinomaly / MoECLIP 写 `history.csv`。
项目自带的渲染器把它们画成 SVG，不引入任何绘图依赖，notebook 里直接显示。
（PatchCore / PaDiM 这类一次成型的记忆库模型没有曲线可画——它们不做梯度训练。）

In [ ]:
from IPython.display import SVG, display
from fabric_defect_hub.reporting.training_curves import (
    discover_training_histories, load_training_curve, render_training_curve_svg,
)

histories = discover_training_histories(["runs", "artifacts/models", "artifacts/training_runs"])
print(f"找到 {len(histories)} 条训练历史")
for index, history in enumerate(histories[-3:], start=1):   # 最近三条
    out = render_training_curve_svg(
        load_training_curve(history),
        pathlib.Path("artifacts/training_curves/notebook") / f"{index:02d}_{history.parent.name}.svg",
    )
    print(history)
    display(SVG(filename=str(out)))

## 7. 评测（可选）

拿刚训好的权重去另一个数据集上打分——跨域那一列的数字就是这么来的。
**`output_dir` 不给就只有图像级指标**：各 adapter 只在拿到写入目录时才产出
`anomaly_map`，像素级 AUROC/AUPRO/IAP 依赖它（见 `docs/DELIVERY_STATUS.md`）。

In [ ]:
weights = run.result.registered_artifact.path
scores = fdh.evaluate(
    MODEL, weights=weights,
    dataset="tilda-400", num_samples=100,
    output_dir="artifacts/anomaly_maps/notebook_eval",   # 去掉这行 = 只有图像级指标
)
scores.metrics

## 附：长时间批量训练不要用 notebook

跑满全部 20 个模型时用批量脚本，它每个模型开独立进程（跑完就释放显存）、
断点续跑、每模型一份日志——notebook 的内核断开就全没了：

```bash
nohup python tools/train_all_models.py --run-id zju-full --accelerator gpu --mode medium \
  > artifacts/train_all.log 2>&1 &
tail -f artifacts/training_runs/zju-full/logs/STFPM.log
```

断了之后续跑：`python tools/train_all_models.py --run-id zju-full --resume --accelerator gpu --mode medium`

环境变量：`FDH_PROGRESS=0` 关掉进度行，`FDH_PROGRESS_INTERVAL=30` 把间隔拉长到 30 秒（日志会小很多）。